## Setup

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm

## Part 1(a)

In [4]:
wages = pd.read_csv("ps1small.csv")

wages['logwage'] = np.log(wages['wage'])
wages['eduxage'] = 'd'+ wages['education'].astype(str) + wages['age'].astype(str)
dummies = pd.get_dummies(wages['eduxage'], dtype = int)
wages = pd.concat([wages, dummies], axis=1)
wages.head()

,wage,education,age,logwage,eduxage,d1226,d1227,d1228,d1229,d1230,...,d1626,d1627,d1628,d1629,d1630,d1726,d1727,d1728,d1729,d1730
0,30000,17,30,10.308953,d1730,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,16000,12,27,9.680344,d1227,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,43500,17,28,10.680516,d1728,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,18000,12,30,9.798127,d1230,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,25000,17,27,10.126631,d1727,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [5]:
model = sm.OLS(endog= wages['logwage'], exog=wages[dummies.columns]).fit()
coefficients = model.params
standard_errors = model.bse
alpha = pd.DataFrame({'Coefficient': coefficients, 'Standard Error': standard_errors})
alpha

,Coefficient,Standard Error
d1226,9.451320,0.090780
d1227,9.300426,0.084980
d1228,9.537856,0.083280
d1229,9.385826,0.085421
d1230,9.628323,0.087737
d1326,9.612120,0.192010
d1327,9.755901,0.192010
d1328,9.378158,0.174517
d1329,9.819982,0.223685
d1330,9.446488,0.202991


alpha is the individual effect of having a given age and education on log wage.

## Part 1(b)

In [6]:
R  = np.zeros((26,30))
q = np.zeros((26,1))

def col(e, a):
    # column index for education level e and age a
    return (e - 12) * 5 + (a - 26)   

#alpha_{e,a-1} - 2*alpha_{e,a} + alpha_{e,a+1} = 0
r = 0
for e in range(12, 18):
    for a in (27, 28, 29):
        R[r, col(e, a-1)] = 1
        R[r, col(e, a)]   = -2
        R[r, col(e, a+1)] = 1
        r += 1

#(alpha_{e-1,27} - alpha_{e-1,26}) - 2*(alpha_{e,27} - alpha_{e,26}) + (alpha_{e+1,27} - alpha_{e+1,26}) = 0
for r in range(18, 22):
    e = 13 + (r - 18)     
    R[r, col(e-1, 27)] = 1
    R[r, col(e-1, 26)] = -1
    R[r, col(e, 27)]   = -2
    R[r, col(e, 26)]   = 2
    R[r, col(e+1, 27)] = 1
    R[r, col(e+1, 26)] = -1

#alpha_{e-1,26} - 2*alpha_{e,26} + alpha_{e+1,26} = 0
for r in range(22, 26):
    e = 13 + (r - 22)
    R[r, col(e-1, 26)] = 1
    R[r, col(e, 26)]   = -2
    R[r, col(e+1, 26)] = 1

In [7]:
wald_test = model.wald_test((R,q), use_f = True)
F = wald_test.fvalue
p_value = wald_test.pvalue
print("F-statistic:", F)
print("p-value:", p_value)

F-statistic: 0.9199279217973075
p-value: 0.5805908886457856


I would not reject the null hypothesis as the p-value (0.58) is not below 0.05 significance level.